# 1_DataPrep
**Purpose:** verify datasets exist, load NASA C-MAPSS FD001 (correct 26-col schema), load PRONOSTIA learning set, save basic scalers and small sanity CSVs.
- data/NASA_CMAPSS/train_FD001.txt
- data/NASA_CMAPSS/test_FD001.txt
- data/NASA_CMAPSS/RUL_FD001.txt
- data/PRONOSTIA/Learning_set/... (extracted from the PHM S3 ZIP)

In [1]:
# Cell 1: imports 
import os, sys
import numpy as np, pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

os.environ["OMP_NUM_THREADS"] = os.environ.get("OMP_NUM_THREADS", "4")
os.environ["MKL_NUM_THREADS"] = os.environ.get("MKL_NUM_THREADS", "4")

print("Python:", sys.version)


Python: 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]


In [5]:
# Cell 2: paths 
ROOT = os.getcwd()
DATA_DIR = os.path.join(ROOT, r"G:\.VIT\SET_PROJECT\PredMaint\Data")
NASA_DIR = os.path.join(DATA_DIR, "NASA_CMAPSS")
PRON_DIR = os.path.join(DATA_DIR, "PRONOSTIA", "Learning_set")
MODEL_DIR = os.path.join(ROOT, r"G:\.VIT\SET_PROJECT\PredMaint\Models")
os.makedirs(MODEL_DIR, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("NASA_DIR:", NASA_DIR)
print("PRON_DIR:", PRON_DIR)
print("MODEL_DIR:", MODEL_DIR)


DATA_DIR: G:\.VIT\SET_PROJECT\PredMaint\Data
NASA_DIR: G:\.VIT\SET_PROJECT\PredMaint\Data\NASA_CMAPSS
PRON_DIR: G:\.VIT\SET_PROJECT\PredMaint\Data\PRONOSTIA\Learning_set
MODEL_DIR: G:\.VIT\SET_PROJECT\PredMaint\Models


In [6]:
# Cell 3: load CMAPSS FD001 (26 columns)
def load_cmapss_fd001(train_path, test_path, rul_path):
    col_names = [
        'unit_number', 'time_in_cycles',
        'op_setting_1', 'op_setting_2', 'op_setting_3',
        'sensor_1','sensor_2','sensor_3','sensor_4','sensor_5',
        'sensor_6','sensor_7','sensor_8','sensor_9','sensor_10',
        'sensor_11','sensor_12','sensor_13','sensor_14','sensor_15',
        'sensor_16','sensor_17','sensor_18','sensor_19','sensor_20',
        'sensor_21'
    ]
    train = pd.read_csv(train_path, sep='\s+', header=None)
    if train.shape[1] != len(col_names):
        raise ValueError(f"train columns {train.shape[1]} != expected {len(col_names)}")
    train.columns = col_names

    test = pd.read_csv(test_path, sep='\s+', header=None)
    if test.shape[1] != len(col_names):
        raise ValueError(f"test columns {test.shape[1]} != expected {len(col_names)}")
    test.columns = col_names

    rul = pd.read_csv(rul_path, sep='\s+', header=None)
    rul.columns = ['RUL']
    return train, test, rul

TRAIN_PATH = os.path.join(NASA_DIR, "train_FD001.txt")
TEST_PATH  = os.path.join(NASA_DIR, "test_FD001.txt")
RUL_PATH   = os.path.join(NASA_DIR, "RUL_FD001.txt")

print("Files exist:", os.path.exists(TRAIN_PATH), os.path.exists(TEST_PATH), os.path.exists(RUL_PATH))

train_df, test_df, rul_df = load_cmapss_fd001(TRAIN_PATH, TEST_PATH, RUL_PATH)
print("train shape:", train_df.shape, "test shape:", test_df.shape, "rul shape:", rul_df.shape)
display(train_df.head())


Files exist: True True True
train shape: (20631, 26) test shape: (13096, 26) rul shape: (100, 1)


,unit_number,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [7]:
# Cell 4: Max cycle per unit and RUL
max_cycle_per_unit = train_df.groupby('unit_number')['time_in_cycles'].max().reset_index().rename(columns={'time_in_cycles':'max_cycle'})
train_df = train_df.merge(max_cycle_per_unit, on='unit_number', how='left')
train_df['RUL'] = train_df['max_cycle'] - train_df['time_in_cycles']
# Small CSV sample for inspection later
train_df.head().to_csv(os.path.join(MODEL_DIR, "cmapss_train_sample.csv"), index=False)
print("Saved cmapss_train_sample.csv")


Saved cmapss_train_sample.csv


In [12]:
# Cell 5: load PRONOSTIA learning set (FEMTO)
import pandas as pd
import glob
import os

def load_pronostia_data(base_dir):
    all_files = glob.glob(os.path.join(base_dir, "*.csv"))
    all_data = []

    for file in all_files:
        try:        
            df = pd.read_csv(file, sep=';', decimal='.', header=None)

            df = df.apply(pd.to_numeric, errors='coerce')
            df.dropna(axis=1, how='all', inplace=True)

            all_data.append(df)

        except Exception as e:
            print(f"Failed reading {file}: {e}")

    # Combine all bearings into one DataFrame
    if all_data:
        data = pd.concat(all_data, ignore_index=True)
        return data
    else:
        print("No valid files loaded.")
        return pd.DataFrame()

# Example usage
pronostia_path = r"G:\.VIT\SET_PROJECT\PredMaint\Data\PRONOSTIA\Learning_set\Bearing1_2"
pronostia_data = load_pronostia_data(pronostia_path)
pronostia_data.head()


,0,1,2,3,4
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN


In [13]:
# Cell 6: save a tiny serialized preview 
if len(signals) > 0:
    preview = pd.DataFrame({
        "label": labels,
        "length": [len(s) for s in signals]
    })
    preview.to_csv(os.path.join(MODEL_DIR, "pronostia_preview.csv"), index=False)
    print("Saved pronostia_preview.csv")
else:
    print("No pronostia signals found; check PRON_DIR path")


Saved pronostia_preview.csv


In [14]:
# Cell 7: save state 
joblib.dump({"train_head": train_df.head(), "rul_head": rul_df.head()}, os.path.join(MODEL_DIR, "data_preview.joblib"))
print("Saved preview joblib")


Saved preview joblib


# 2_FeatureEngineering
**Purpose:** sliding window features for C-MAPSS, time & frequency features for PRONOSTIA, save feature CSVs for model training.


In [15]:
# Cell 2: load saved raw data preview 
def load_cmapss_fd001_quiet(train_path, test_path, rul_path):
    col_names = [
        'unit_number', 'time_in_cycles',
        'op_setting_1', 'op_setting_2', 'op_setting_3'
    ] + [f'sensor_{i}' for i in range(1,22)]
    train = pd.read_csv(train_path, sep='\s+', header=None)
    train.columns = col_names
    test = pd.read_csv(test_path, sep='\s+', header=None)
    test.columns = col_names
    rul = pd.read_csv(rul_path, sep='\s+', header=None); rul.columns=['RUL']
    return train, test, rul

train_df, test_df, rul_df = load_cmapss_fd001_quiet(TRAIN_PATH, TEST_PATH, RUL_PATH)
# recompute max cycles and RUL
train_df = train_df.merge(train_df.groupby('unit_number')['time_in_cycles'].max().reset_index().rename(columns={'time_in_cycles':'max_cycle'}), on='unit_number', how='left')
train_df['RUL'] = train_df['max_cycle'] - train_df['time_in_cycles']
SENSOR_COLS = [c for c in train_df.columns if c.startswith('sensor_')]
print("Sensors:", len(SENSOR_COLS))


Sensors: 21


In [16]:
# Cell 3: scaling sensors 
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_df[SENSOR_COLS] = scaler.fit_transform(train_df[SENSOR_COLS])
test_df[SENSOR_COLS]  = scaler.transform(test_df[SENSOR_COLS])
joblib.dump(scaler, os.path.join(MODEL_DIR, "cmapss_scaler.joblib"))
print("Saved cmapss_scaler.joblib")


Saved cmapss_scaler.joblib


In [17]:
# Cell 4: sliding window aggregation for C-MAPSS
def sliding_window_features(df, sensor_cols, window_size=30, step=5, agg_funcs=['mean','std','min','max','median']):
    rows=[]
    for uid, g in df.groupby('unit_number'):
        g = g.sort_values('time_in_cycles').reset_index(drop=True)
        max_cycle = g['time_in_cycles'].max()
        n = len(g)
        for start in range(0, n - window_size + 1, step):
            win = g.iloc[start:start+window_size]
            end_cycle = int(win['time_in_cycles'].iloc[-1])
            feat = {"unit_number":uid, "end_cycle": end_cycle}
            for s in sensor_cols:
                arr = win[s].values
                for f in agg_funcs:
                    key = f"{s}_{f}"
                    if hasattr(np, f):
                        feat[key] = float(getattr(np, f)(arr))
                    else:
                        feat[key] = float(pd.Series(arr).agg(f))
            rul = max_cycle - end_cycle
            feat['RUL'] = rul
            feat['RUL_cap'] = min(rul, 125)
            rows.append(feat)
    return pd.DataFrame(rows)

df_windows = sliding_window_features(train_df, SENSOR_COLS, window_size=30, step=5)
print("Windowed features:", df_windows.shape)
df_windows.to_csv(os.path.join(MODEL_DIR, "cmapss_windows.csv"), index=False)
print("Saved cmapss_windows.csv")


Windowed features: (3586, 109)
Saved cmapss_windows.csv


In [23]:
with open(r"G:\.VIT\SET_PROJECT\PredMaint\Data\PRONOSTIA\Learning_set\Bearing1_1\acc_00001.csv", 'r') as f:
    for _ in range(5):
        print(f.readline().strip())

9,39,39,65664,0.552,-0.146
9,39,39,65703,0.501,-0.48
9,39,39,65742,0.138,0.435
9,39,39,65781,-0.423,0.24
9,39,39,65820,-0.802,0.02


In [25]:
# Cell 5: PRONOSTIA features - time and frequency bands
from scipy.stats import kurtosis, skew
from scipy import signal
def extract_time_features(sig):
    sig = np.asarray(sig)
    return {
        "mean": float(np.mean(sig)),
        "std": float(np.std(sig)),
        "rms": float(np.sqrt(np.mean(sig**2))),
        "peak": float(np.max(np.abs(sig))),
        "kurtosis": float(kurtosis(sig)),
        "skew": float(skew(sig))
    }

def extract_freq_bands(sig, fs=20000, bands=[(0,500),(500,1500),(1500,3000)]):
    f, Pxx = signal.welch(sig, fs=fs, nperseg=2048)
    d = {}
    for lo,hi in bands:
        mask = (f>=lo)&(f<hi)
        d[f"band_{lo}_{hi}"] = float(Pxx[mask].sum()) if mask.any() else 0.0
    return d

# load signals from 1_DataPrep preview if needed; otherwise load fresh
from glob import glob
signal_files = []
if os.path.exists(PRON_DIR):
    for fol in sorted(os.listdir(PRON_DIR)):
        fpath = os.path.join(PRON_DIR, fol)
        if os.path.isdir(fpath):
            for f in sorted(os.listdir(fpath)):
                if f.lower().endswith(('.csv','.txt')):
                    signal_files.append(os.path.join(fpath, f))

print("Pron files found:", len(signal_files))

feat_rows = []
labels = []

for p in signal_files:
    # Skip non-acceleration files (like temp_)
    if not os.path.basename(p).lower().startswith("acc_"):
        continue

    try:
        df = pd.read_csv(p, header=None)
        
        # Only last two numeric columns (acc1, acc2)
        sig = df.iloc[:, -2:].apply(pd.to_numeric, errors='coerce').to_numpy().flatten()
        sig = sig[~np.isnan(sig)]  # remove NaNs

        if len(sig) == 0:
            print(f"Empty numeric data after cleaning: {p}")
            continue

        # Extract features
        tf = extract_time_features(sig)
        ff = extract_freq_bands(sig)
        feat_rows.append({**tf, **ff})
        labels.append(os.path.basename(os.path.dirname(p)))

    except Exception as e:
        print("Read fail", p, e)

# Save
feats_pron = pd.DataFrame(feat_rows).fillna(0)
feats_pron['label'] = labels
feats_pron.to_csv(os.path.join(MODEL_DIR, "pronostia_features.csv"), index=False)
print("Saved pronostia_features.csv", feats_pron.shape)


Pron files found: 8384
✅ Saved pronostia_features.csv (7534, 10)


In [26]:
# Cell 6: summary saved
print("C-MAPSS windows csv and PRON features saved in", MODEL_DIR)

C-MAPSS windows csv and PRON features saved in G:\.VIT\SET_PROJECT\PredMaint\Models


# 3_Models
**Purpose:** train baseline ML (RF + CPU XGBoost), train RNN (LSTM/GRU) for RUL, train 1D-CNN for PRONOSTIA. All CPU-only (no GPU).

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report
import xgboost as xgb
# set CPU threaded env if desired
os.environ["OMP_NUM_THREADS"] = os.environ.get("OMP_NUM_THREADS","4")
os.makedirs(MODEL_DIR, exist_ok=True)
print("xgboost version:", xgb.__version__)

xgboost version: 2.1.1


In [33]:
# Cell 2: load C-MAPSS windows and train
MODEL_DIR = os.path.join(ROOT, r"G:\.VIT\SET_PROJECT\PredMaint\Models")
df_windows = pd.read_csv(os.path.join(MODEL_DIR, "cmapss_windows.csv")) 
feature_cols = [c for c in df_windows.columns if c not in ['unit_number','end_cycle','RUL','RUL_cap']]
X = df_windows[feature_cols].fillna(0).values
y = df_windows['RUL_cap'].values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print("Shapes:", X_train.shape, X_val.shape)


Shapes: (2868, 105) (718, 105)


In [34]:
# Cell 3: RandomForest baseline (RUL)
rf = RandomForestRegressor(n_estimators=150, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_val)
print("RF RMSE:", np.sqrt(mean_squared_error(y_val, pred_rf)), "MAE:", mean_absolute_error(y_val, pred_rf))
joblib.dump(rf, os.path.join(MODEL_DIR, "rf_rul.joblib"))


RF RMSE: 11.83383789056009 MAE: 8.400594243268339


['G:\\.VIT\\SET_PROJECT\\PredMaint\\Models\\rf_rul.joblib']

In [40]:
# Cell 4: CPU XGBoost baseline (RUL) 
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import xgboost as xgb

# CPU-based XGBoost baseline (RUL)
xgb_model = xgb.XGBRegressor(
    tree_method='hist',
    predictor='cpu_predictor',
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    n_jobs=4
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

pred_xgb = xgb_model.predict(X_val)
print("XGB RMSE:", np.sqrt(mean_squared_error(y_val, pred_xgb)),
      "MAE:", mean_absolute_error(y_val, pred_xgb))


XGB RMSE: 10.805677080383596 MAE: 7.311689376831055


In [41]:
# Cell 5: LSTM sequence model for RUL 
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

NASA_DIR = os.path.join(DATA_DIR, "NASA_CMAPSS")
def load_scaled_train(train_path):
    col_names = [
        'unit_number', 'time_in_cycles',
        'op_setting_1', 'op_setting_2', 'op_setting_3'
    ] + [f'sensor_{i}' for i in range(1,22)]
    df = pd.read_csv(train_path, sep='\s+', header=None)
    df.columns = col_names
    # recompute RUL
    df = df.merge(df.groupby('unit_number')['time_in_cycles'].max().reset_index().rename(columns={'time_in_cycles':'max_cycle'}), on='unit_number', how='left')
    df['RUL'] = df['max_cycle'] - df['time_in_cycles']
    # load scaler
    scaler = joblib.load(os.path.join(MODEL_DIR,"cmapss_scaler.joblib"))
    df[[c for c in df.columns if c.startswith('sensor_')]] = scaler.transform(df[[c for c in df.columns if c.startswith('sensor_')]])
    return df

train_df = load_scaled_train(os.path.join(NASA_DIR,"train_FD001.txt"))
SENSOR_COLS = [c for c in train_df.columns if c.startswith('sensor_')]

SEQ_LEN = 30
def create_sequences(df, sensor_cols, seq_len=SEQ_LEN):
    Xs, ys = [], []
    for uid, g in df.groupby('unit_number'):
        g = g.sort_values('time_in_cycles').reset_index(drop=True)
        max_cycle = g['time_in_cycles'].max()
        for i in range(len(g)):
            if i+1 >= seq_len:
                seq = g.loc[i-seq_len+1:i, sensor_cols].values
                Xs.append(seq)
                ys.append(min(max_cycle - g.loc[i,'time_in_cycles'], 125))
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(train_df, SENSOR_COLS, SEQ_LEN)
Xtr, Xv, ytr, yv = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)
print("Sequence shapes:", Xtr.shape, Xv.shape)

def build_rnn(seq_len, n_feats, units=128):
    model = Sequential()
    model.add(Bidirectional(GRU(units, return_sequences=False), input_shape=(seq_len, n_feats)))
    model.add(Dropout(0.2))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1, activation='linear'))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

rnn = build_rnn(SEQ_LEN, len(SENSOR_COLS), units=128)
rnn.summary()
es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
mc = ModelCheckpoint(os.path.join(MODEL_DIR,"rnn_rul.h5"), save_best_only=True)
rnn.fit(Xtr, ytr, validation_data=(Xv,yv), epochs=40, batch_size=64, callbacks=[es,mc], verbose=2)
pred_rnn = rnn.predict(Xv).ravel()
print("RNN RMSE:", np.sqrt(mean_squared_error(yv, pred_rnn)))
rnn.save(os.path.join(MODEL_DIR,"rul_rnn_final.h5"))


Sequence shapes: (14184, 30, 21) (3547, 30, 21)
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirection  (None, 256)               115968    
 al)                                                             
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense (Dense)               (None, 64)                16448     
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 132481 (517.50 KB)
Trainable params: 132481 (517.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/40
222/222 - 16s - loss: 1

# 4_Evaluation_Report
**Purpose:** load saved models, compute final metrics, produce plots, save interim CSV/JSON.

In [42]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, accuracy_score, f1_score
MODEL_DIR = os.path.join(ROOT, r"G:\.VIT\SET_PROJECT\PredMaint\Models")
print("MODEL_DIR:", MODEL_DIR)

MODEL_DIR: G:\.VIT\SET_PROJECT\PredMaint\Models


In [43]:
# Cell 2: evaluate RF and XGBoost RUL
rf = joblib.load(os.path.join(MODEL_DIR,"rf_rul.joblib"))
xgb_model = None
if os.path.exists(os.path.join(MODEL_DIR,"xgb_rul.json")):
    try:
        import xgboost as xgb
        xgb_model = xgb.XGBRegressor()
        xgb_model.load_model(os.path.join(MODEL_DIR,"xgb_rul.json"))
    except Exception as e:
        print("Failed loading xgb model:", e)

# load windowed data
df_windows = pd.read_csv(os.path.join(MODEL_DIR,"cmapss_windows.csv"))
feature_cols = [c for c in df_windows.columns if c not in ['unit_number','end_cycle','RUL','RUL_cap']]
X = df_windows[feature_cols].fillna(0).values
y = df_windows['RUL_cap'].values
# make a simple split to mimic training split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size=0.2, random_state=42)

pred_rf = rf.predict(X_val)
metrics_rf = {"RMSE": float(np.sqrt(mean_squared_error(y_val,pred_rf))), "MAE": float(mean_absolute_error(y_val,pred_rf))}
print("RF", metrics_rf)
metrics_xgb = None
if xgb_model:
    pred_xgb = xgb_model.predict(X_val)
    metrics_xgb = {"RMSE": float(np.sqrt(mean_squared_error(y_val,pred_xgb))), "MAE": float(mean_absolute_error(y_val,pred_xgb))}
    print("XGB", metrics_xgb)


RF {'RMSE': 11.83383789056009, 'MAE': 8.400594243268339}


In [45]:
# Cell 3: evaluate RNN (load and test sequences)
from tensorflow.keras.models import load_model
if os.path.exists(os.path.join(MODEL_DIR,"rul_rnn_final.h5")):
    rnn = load_model(os.path.join(MODEL_DIR,"rul_rnn_final.h5"))
    # prepare sequences again (reuse code from Models notebook or load saved Xv,yv) - for brevity we load the last saved Xv,yv if present

    TRAIN_PATH = os.path.join(NASA_DIR, "train_FD001.txt")
    def create_sequences_min(train_path, seq_len=30):
        col_names = [
            'unit_number','time_in_cycles','op_setting_1','op_setting_2','op_setting_3'
        ] + [f'sensor_{i}' for i in range(1,22)]
        df = pd.read_csv(train_path, sep='\s+', header=None)
        df.columns = col_names
        df = df.merge(df.groupby('unit_number')['time_in_cycles'].max().reset_index().rename(columns={'time_in_cycles':'max_cycle'}), on='unit_number', how='left')
        df['RUL'] = df['max_cycle'] - df['time_in_cycles']
        scaler = joblib.load(os.path.join(MODEL_DIR,"cmapss_scaler.joblib"))
        df[[c for c in df.columns if c.startswith('sensor_')]] = scaler.transform(df[[c for c in df.columns if c.startswith('sensor_')]])
        Xs, ys = [], []
        for uid, g in df.groupby('unit_number'):
            g = g.sort_values('time_in_cycles').reset_index(drop=True)
            max_cycle = g['time_in_cycles'].max()
            for i in range(len(g)):
                if i+1>=seq_len:
                    seq = g.loc[i-seq_len+1:i, [c for c in df.columns if c.startswith('sensor_')]].values
                    Xs.append(seq); ys.append(min(max_cycle - g.loc[i,'time_in_cycles'],125))
        Xs = np.array(Xs); ys=np.array(ys)
        return train_test_split(Xs, ys, test_size=0.2, random_state=42)
    try:
        Xtr, Xv_rnn, ytr, yv_rnn = create_sequences_min(TRAIN_PATH, seq_len=30)
        pred = rnn.predict(Xv_rnn).ravel()
        print("RNN RMSE:", float(np.sqrt(mean_squared_error(yv_rnn, pred))))
    except Exception as e:
        print("Failed to re-eval RNN:", e)
else:
    print("RNN model not found.")


111/111 [==============================] - 3s 20ms/step
RNN RMSE: 4.055018652180437


In [47]:
# Cell 3B: Train PRONOSTIA classifier (Random Forest)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib, os, pandas as pd

feats_path = os.path.join(MODEL_DIR, "pronostia_features.csv")
if os.path.exists(feats_path):
    feats_df = pd.read_csv(feats_path)
    X = feats_df.drop(columns=['label']).values
    y_labels = feats_df['label']

    le = LabelEncoder()
    y = le.fit_transform(y_labels)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    clf = RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42
    )
    clf.fit(X_train, y_train)

    pred = clf.predict(X_val)
    print("Training complete. Validation report:")
    print(classification_report(y_val, pred, zero_division=0))

    # Save model + label encoder
    joblib.dump(clf, os.path.join(MODEL_DIR, "rf_pronostia.joblib"))
    joblib.dump(le, os.path.join(MODEL_DIR, "pronostia_label_encoder.joblib"))
    print("Model and encoder saved.")
else:
    print("pronostia_features.csv not found. Run feature extraction first.")

Training complete. Validation report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       561
           1       0.95      0.99      0.97       174
           2       0.99      0.95      0.97       182
           3       0.98      0.99      0.98       159
           4       0.91      0.86      0.89       103
           5       0.96      0.96      0.96       328

    accuracy                           0.97      1507
   macro avg       0.96      0.96      0.96      1507
weighted avg       0.97      0.97      0.97      1507

✅ Model and encoder saved.


In [48]:
# Cell 4: PRONOSTIA classifier evaluation
if os.path.exists(os.path.join(MODEL_DIR,"rf_pronostia.joblib")):
    clf = joblib.load(os.path.join(MODEL_DIR,"rf_pronostia.joblib"))
    feats_df = pd.read_csv(os.path.join(MODEL_DIR,"pronostia_features.csv"))
    X = feats_df.drop(columns=['label']).values
    le = joblib.load(os.path.join(MODEL_DIR,"pronostia_label_encoder.joblib"))
    y = le.transform(feats_df['label'])
    Xtr, Xv, ytr, yv = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    pred = clf.predict(Xv)
    print("PRON classification report:")
    print(classification_report(yv, pred, zero_division=0))
else:
    print("Pron classifier not found.")


PRON classification report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       561
           1       0.95      0.99      0.97       174
           2       0.99      0.95      0.97       182
           3       0.98      0.99      0.98       159
           4       0.91      0.86      0.89       103
           5       0.96      0.96      0.96       328

    accuracy                           0.97      1507
   macro avg       0.96      0.96      0.96      1507
weighted avg       0.97      0.97      0.97      1507



In [50]:
# Cell 5: Save interim report
import json
report = {
    "rf_rul": metrics_rf,
    "xgb_rul": metrics_xgb,
}
with open(os.path.join(MODEL_DIR,"semester1_report.json"),"w") as f:
    json.dump(report, f, indent=2)
print("Saved semester1_report.json")


Saved semester1_report.json
